In [1]:
!pip install -q delta-spark pyspark

In [2]:
import delta
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Part_8_merge") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(delta.__version__)
print("Spark session ready")

4.2.0
Spark session ready


In [3]:
visits_data = [
(1,1001,201,"2024-03-01","Completed",2),
(2,1002,202,"2024-03-01","Completed",1),
(3,1003,203,"2024-03-02","Completed",3),
(4,1004,204,"2024-03-02","Pending",1),
(5,1005,206,"2024-03-03","Completed",2),
(6,1006,205,"2024-03-03","Completed",4),
(7,1007,207,"2024-03-04","Cancelled",1),
(8,1008,208,"2024-03-04","Completed",2),
(9,1009,201,"2024-03-05","Completed",1),
(10,1010,202,"2024-03-05","Completed",2),
(11,1011,205,"2024-03-06","Pending",3),
(12,1012,204,"2024-03-06","Completed",1),
(13,1013,203,"2024-03-07","Completed",2),
(14,1014,201,"2024-03-07","Completed",3),
(15,1015,210,"2024-03-08","Completed",1),
(16,1016,207,"2024-03-08","Cancelled",2),
(17,1017,209,"2024-03-09","Completed",4),
(18,1018,206,"2024-03-09","Completed",2),
(19,1019,209,"2024-03-10","Completed",3),
(20,1020,206,"2024-03-10","Pending",2),
(21,1001,205,"2024-03-11","Completed",3),
(22,1003,208,"2024-03-11","Completed",2),
(23,1006,201,"2024-03-12","Completed",1),
(24,1009,210,"2024-03-12","Completed",2),
(25,1014,202,"2024-03-13","Completed",1)
]
visits_columns = [
"visit_id","patient_id","doctor_id","visit_date","visit_status","tests_count"
]
visits_df = spark.createDataFrame(visits_data, visits_columns)

In [4]:
daily_visits_data = [
(26,1002,201,"2024-03-14","Completed",2),
(27,1005,210,"2024-03-14","Completed",1),
(11,1011,205,"2024-03-06","Completed",3),
(20,1020,206,"2024-03-10","Completed",2),
(28,1018,203,"2024-03-15","Pending",2)
]

daily_visits_columns = [
"visit_id","patient_id","doctor_id","visit_date","visit_status","tests_count"
]

daily_visits_df = spark.createDataFrame(daily_visits_data, daily_visits_columns)

In [8]:
#Exercise 71
visits_df.write.format("delta").mode("overwrite").save("/tmp/visits")
visits_df.createOrReplaceTempView("visits")
spark.sql("""
create table if not exists visits_target
using delta
location '/tmp/visits'
""")

DataFrame[]

In [9]:
#Exercise 72
spark.sql("""
INSERT INTO visits_target
SELECT * FROM visits
""")

DataFrame[]

In [10]:
#Exercise 73
daily_visits_df.createOrReplaceTempView("daily_visits_updates")

In [11]:
#Exercise 74
spark.sql("""
merge into visits_target t
using daily_visits_updates s
on t.visit_id = s.visit_id
when matched then update set *
when not matched then insert *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [13]:
#Exercise 75
spark.sql("""
merge into visits_target t
using daily_visits_updates s
on t.visit_id = s.visit_id
when matched then update set
t.visit_id = s.visit_id,
t.patient_id = s.patient_id,
t.doctor_id = s.doctor_id,
t.visit_date = s.visit_date,
t.visit_status = s.visit_status,
t.tests_count = s.tests_count
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [15]:
#Exercise 76
spark.sql("""
merge into visits_target t
using daily_visits_updates s
on t.visit_id = s.visit_id
when not matched then insert (t.visit_id, t.patient_id, t.doctor_id, t.visit_date, t.visit_status, t.tests_count)
values (s.visit_id, s.patient_id, s.doctor_id, s.visit_date, s.visit_status, s.tests_count)
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [ ]:
#Exercise 77
spark.sql("""
SELECT visit_id, visit_status
FROM visits_target
WHERE visit_id IN (11, 20)
""").show()

In [ ]:
#Exercise 78
spark.sql("""
SELECT *
FROM visits_target
WHERE visit_id IN (26, 27, 28)
""").show()

In [19]:
#Exercise 79
spark.sql("""
describe history visits_target
""").show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      4|2026-05-06 14:14:...|  NULL|    NULL|               MERGE|{predicate -> ["(...|NULL|    NULL|     NULL|          3|  Serializable|        false|{numTargetRowsCop...|        NULL|Apache-Spark/4.0....|
|      3|2026-05-06 14:13:...|  NULL|    NULL|               MERGE|{predicate -> ["(...|NULL|    NULL|     NULL|          2|  Serializable|        false|{numTargetR

In [ ]:
#Exercise 80


MERGE is used for incremental loads to handle both inserts and updates in a single operation.
It avoids duplicate logic by combining UPDATE and INSERT (UPSERT).
It ensures Delta tables stay consistent with incoming daily changes.